In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_score
from imblearn.over_sampling import SMOTE

In [3]:
outcomes = pd.read_csv('outcomes.csv')
projects = pd.read_csv('projects.csv')

In [4]:
binary_cols = ['fully_funded', 'at_least_1_teacher_referred_donor', 'great_chat',
               'three_or_more_non_teacher_referred_donors', 
               'one_non_teacher_referred_donor_giving_100_plus',
               'donation_from_thoughtful_donor', 'at_least_1_green_donation']


In [5]:
for col in binary_cols:
    outcomes[col] = outcomes[col].map({'t': 1, 'f': 0})

# Merge datasets
df = projects.merge(outcomes[['projectid', 'fully_funded']], on='projectid', how='inner')

In [6]:
drop_cols = ['projectid', 'teacher_acctid', 'schoolid', 'school_ncesid', 'school_city', 
             'school_zip', 'school_district', 'school_county', 'school_latitude', 
             'school_longitude']
df = df.drop(columns=drop_cols, errors='ignore')
df

,school_state,school_metro,school_charter,school_magnet,school_year_round,school_nlns,school_kipp,school_charter_ready_promise,teacher_prefix,teacher_teach_for_america,...,poverty_level,grade_level,fulfillment_labor_materials,total_price_excluding_optional_support,total_price_including_optional_support,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded
0,IL,suburban,f,f,f,f,f,f,Mrs.,f,...,moderate poverty,Grades 3-5,30.0,444.36,522.78,7.0,f,f,2013-12-31,1
1,ID,urban,f,f,f,f,f,f,Mrs.,f,...,high poverty,Grades 3-5,30.0,233.24,274.40,30.0,f,f,2013-12-31,0
2,NH,suburban,f,f,f,f,f,f,Mrs.,f,...,moderate poverty,Grades 6-8,30.0,285.09,335.40,230.0,f,f,2013-12-31,0
3,VA,urban,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades PreK-2,30.0,232.94,274.05,18.0,f,f,2013-12-31,0
4,IL,urban,f,t,f,f,f,f,Mr.,f,...,highest poverty,Grades 6-8,30.0,513.41,604.01,70.0,t,f,2013-12-31,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619321,NY,urban,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades PreK-2,NaN,231.00,281.71,0.0,f,f,2002-09-17,1
619322,NY,urban,f,f,f,f,f,f,Mr.,f,...,highest poverty,Grades 9-12,NaN,1129.00,1376.83,0.0,f,f,2002-09-17,1
619323,NY,urban,f,t,f,f,f,f,Ms.,f,...,moderate poverty,Grades 3-5,NaN,125.00,152.44,0.0,f,f,2002-09-16,1
619324,NY,NaN,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades 9-12,NaN,125.00,152.44,0.0,f,f,2002-09-13,1


In [7]:
categorical_cols = [
    'school_state', 'school_metro', 'teacher_prefix', 'primary_focus_area',
    'primary_focus_subject', 'secondary_focus_area', 'secondary_focus_subject',
    'resource_type', 'poverty_level', 'grade_level',
    'school_charter', 'school_magnet', 'school_year_round',
    'school_nlns', 'school_kipp', 'school_charter_ready_promise',
    'teacher_teach_for_america', 'teacher_ny_teaching_fellow',
    'eligible_double_your_impact_match', 'eligible_almost_home_match'
]

In [8]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df.fillna(0, inplace=True)

In [9]:
df['date_posted'] = pd.to_datetime(df['date_posted'])
df.sort_values('date_posted', inplace=True)
df.drop(columns=['date_posted'], inplace=True)

In [10]:
# -----------------------
# 3. Incremental Feature Selection
# -----------------------

In [11]:
split = int(0.8 * len(df))
train_df, test_df = df.iloc[:split], df.iloc[split:]

X_train, X_test = df.drop('fully_funded', axis=1), df.drop('fully_funded', axis=1)
y_train, y_test = df['fully_funded'], df['fully_funded']

In [12]:
available_features = list(X_train.columns)


In [13]:
def incremental_feature_selection(model, model_name):
    selected_features = []
    best_precision = 0.0
    print(f"\n--- Incremental Feature Selection ({model_name}) ---")

    for feature in available_features:
        trial_features = selected_features + [feature]

        model.fit(X_train[trial_features], y_train)
        pred_probs = model.predict_proba(X_test[trial_features])[:, 1]

        # Bottom 10% selection
        threshold = np.percentile(pred_probs, 10)
        preds_bottom_10 = (pred_probs <= threshold).astype(int)
        y_test_negative = (y_test == 0).astype(int)

        precision_bottom_10 = precision_score(y_test_negative, preds_bottom_10)

        print(f"Feature: {feature:<40} | Bottom 10% Precision: {precision_bottom_10:.4f}", end=' ')

        if precision_bottom_10 >= best_precision:
            selected_features.append(feature)
            best_precision = precision_bottom_10
            print("✅ Added")
        else:
            print("❌ Not added")

    print(f"\nFinal Selected Features ({model_name}):")
    print(selected_features)

    # Final evaluation
    final_model = model
    final_model.fit(X_train[selected_features], y_train)
    final_pred_probs = final_model.predict_proba(X_test[selected_features])[:, 1]
    final_preds = final_model.predict(X_test[selected_features])

    roc_auc = roc_auc_score(y_test, final_pred_probs)
    class_report = classification_report(y_test, final_preds)

    final_threshold = np.percentile(final_pred_probs, 10)
    final_preds_bottom_10 = (final_pred_probs <= final_threshold).astype(int)
    final_precision_bottom_10 = precision_score(y_test_negative, final_preds_bottom_10)

    print(f"\n--- Final {model_name} Model Evaluation ---")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("\nClassification Report:\n", class_report)
    print(f"\nPrecision on Bottom 10% least likely funded: {final_precision_bottom_10:.4f}")

In [63]:
# -----------------------
# 4. Final Model Evaluation
# -----------------------


In [65]:
rf_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
incremental_feature_selection(rf_model, "Random Forest")


--- Incremental Feature Selection (Random Forest) ---
Feature: fulfillment_labor_materials              | Bottom 10% Precision: 0.3748 ✅ Added
Feature: total_price_excluding_optional_support   | Bottom 10% Precision: 0.5449 ✅ Added
Feature: total_price_including_optional_support   | Bottom 10% Precision: 0.5386 ❌ Not added
Feature: students_reached                         | Bottom 10% Precision: 0.5423 ❌ Not added
Feature: school_state_AL                          | Bottom 10% Precision: 0.5464 ✅ Added
Feature: school_state_AR                          | Bottom 10% Precision: 0.5413 ❌ Not added
Feature: school_state_AZ                          | Bottom 10% Precision: 0.5408 ❌ Not added
Feature: school_state_CA                          | Bottom 10% Precision: 0.5482 ✅ Added
Feature: school_state_CO                          | Bottom 10% Precision: 0.5427 ❌ Not added
Feature: school_state_CT                          | Bottom 10% Precision: 0.5346 ❌ Not added
Feature: school_state_DC       

In [66]:
dt_model = DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42)
incremental_feature_selection(dt_model, "Decision Tree")


--- Incremental Feature Selection (Decision Tree) ---
Feature: fulfillment_labor_materials              | Bottom 10% Precision: 0.3748 ✅ Added
Feature: total_price_excluding_optional_support   | Bottom 10% Precision: 0.5149 ✅ Added
Feature: total_price_including_optional_support   | Bottom 10% Precision: 0.5136 ❌ Not added
Feature: students_reached                         | Bottom 10% Precision: 0.5148 ❌ Not added
Feature: school_state_AL                          | Bottom 10% Precision: 0.5149 ✅ Added
Feature: school_state_AR                          | Bottom 10% Precision: 0.5149 ❌ Not added
Feature: school_state_AZ                          | Bottom 10% Precision: 0.5149 ✅ Added
Feature: school_state_CA                          | Bottom 10% Precision: 0.4942 ❌ Not added
Feature: school_state_CO                          | Bottom 10% Precision: 0.5149 ✅ Added
Feature: school_state_CT                          | Bottom 10% Precision: 0.5149 ✅ Added
Feature: school_state_DC               

In [15]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [16]:
available_features_sm = list(X_train_sm.columns)

In [17]:
def incremental_selection_smote(model, X_tr, y_tr, X_te, y_te, model_name):
    selected_features = []
    best_precision = 0.0
    print(f"\n--- Incremental Feature Selection ({model_name}, SMOTE-balanced) ---")

    for feature in available_features_sm:
        trial_features = selected_features + [feature]

        model.fit(X_tr[trial_features], y_tr)
        pred_probs = model.predict_proba(X_te[trial_features])[:, 1]

        threshold = np.percentile(pred_probs, 10)
        preds_bottom_10 = (pred_probs <= threshold).astype(int)
        y_te_negative = (y_te == 0).astype(int)

        precision_bottom_10 = precision_score(y_te_negative, preds_bottom_10)

        print(f"Feature: {feature:<40} | Bottom 10% Precision: {precision_bottom_10:.4f}", end=' ')

        if precision_bottom_10 >= best_precision:
            selected_features.append(feature)
            best_precision = precision_bottom_10
            print("✅ Added")
        else:
            print("❌ Not added")

    print(f"\nFinal Selected Features ({model_name}):")
    print(selected_features)

    # Final evaluation
    final_model = model
    final_model.fit(X_tr[selected_features], y_tr)
    final_pred_probs = final_model.predict_proba(X_te[selected_features])[:, 1]
    final_preds = final_model.predict(X_te[selected_features])

    roc_auc = roc_auc_score(y_te, final_pred_probs)
    class_report = classification_report(y_te, final_preds)

    final_threshold = np.percentile(final_pred_probs, 10)
    final_preds_bottom_10 = (final_pred_probs <= final_threshold).astype(int)
    final_precision_bottom_10 = precision_score(y_te_negative, final_preds_bottom_10)

    print(f"\n--- Final {model_name} Model Evaluation ---")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("\nClassification Report:\n", class_report)
    print(f"\nPrecision on Bottom 10% least likely funded: {final_precision_bottom_10:.4f}")

In [18]:
rf_model_smote = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
incremental_selection_smote(rf_model_smote, X_train_sm, y_train_sm, X_test, y_test, "Random Forest")



--- Incremental Feature Selection (Random Forest, SMOTE-balanced) ---
Feature: fulfillment_labor_materials              | Bottom 10% Precision: 0.3748 ✅ Added
Feature: total_price_excluding_optional_support   | Bottom 10% Precision: 0.5340 ✅ Added
Feature: total_price_including_optional_support   | Bottom 10% Precision: 0.5362 ✅ Added
Feature: students_reached                         | Bottom 10% Precision: 0.5350 ❌ Not added
Feature: school_state_AL                          | Bottom 10% Precision: 0.5352 ❌ Not added
Feature: school_state_AR                          | Bottom 10% Precision: 0.5343 ❌ Not added
Feature: school_state_AZ                          | Bottom 10% Precision: 0.5340 ❌ Not added
Feature: school_state_CA                          | Bottom 10% Precision: 0.5378 ✅ Added
Feature: school_state_CO                          | Bottom 10% Precision: 0.5301 ❌ Not added
Feature: school_state_CT                          | Bottom 10% Precision: 0.5308 ❌ Not added
Feature: school

In [19]:
dt_model_smote = DecisionTreeClassifier(max_depth=5, random_state=42)
incremental_selection_smote(dt_model_smote, X_train_sm, y_train_sm, X_test, y_test, "Decision Tree")


--- Incremental Feature Selection (Decision Tree, SMOTE-balanced) ---
Feature: fulfillment_labor_materials              | Bottom 10% Precision: 0.3748 ✅ Added
Feature: total_price_excluding_optional_support   | Bottom 10% Precision: 0.5111 ✅ Added
Feature: total_price_including_optional_support   | Bottom 10% Precision: 0.4992 ❌ Not added
Feature: students_reached                         | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_AL                          | Bottom 10% Precision: 0.5087 ❌ Not added
Feature: school_state_AR                          | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_AZ                          | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_CA                          | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_CO                          | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_CT                          | Bottom 10% Precision: 0.5111 ✅ Added
Feature: school_state_DC       